# Suspend duplicate Lemma among new cards only (AnkiConnect)

This notebook:
- scans `is:new` cards
- groups notes by normalized `Lemma`
- for groups with duplicates (2+ notes), suspends those new cards
- tags duplicate notes with `~suspended_duplicate_lemma_review_is_new`
- writes sibling links to `Link` as `[Title|nid...]`

Run first with `DRY_RUN = True`.

In [ ]:
import json
import re
import urllib.request
from collections import defaultdict

ANKI_CONNECT_URL = "http://127.0.0.1:8765"
ANKI_CONNECT_VERSION = 6

DECK_SCOPE = 'deck:"Japanese Media"'
NEW_QUERY = f"{DECK_SCOPE} is:new"
NEW_ELIGIBLE_QUERY = f"{NEW_QUERY} -is:suspended -tag:~suspended_duplicate_lemma_review_is_new"

LEMMA_FIELD_PREFERRED = "Lemma"
LEMMA_FIELD_ALIASES = ["Lemma", "lemma", "Word", "Expression"]
SUBTITLE_FIELD_PREFERRED = "Subtitle"
SUBTITLE_FIELD_ALIASES = ["Subtitle", "subtitle", "Sentence", "Context"]
LINK_FIELD = "Link"
TAG_DUPLICATE_NEW = "~suspended_duplicate_lemma_review_is_new"
DRY_RUN = True

TAG_RE = re.compile(r"<[^>]+>")

def invoke(action, **params):
    payload = json.dumps({"action": action, "version": ANKI_CONNECT_VERSION, "params": params}).encode("utf-8")
    req = urllib.request.Request(ANKI_CONNECT_URL, data=payload, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req) as resp:
        body = json.loads(resp.read().decode("utf-8"))
    if body.get("error"):
        raise RuntimeError(f"AnkiConnect error in {action}: {body['error']}")
    return body.get("result")

def chunked(items, size=500):
    for i in range(0, len(items), size):
        yield items[i:i + size]

def get_cards_info(card_ids):
    out = []
    for batch in chunked(card_ids, 500):
        out.extend(invoke("cardsInfo", cards=batch))
    return out

def get_notes_info(note_ids):
    out = []
    for batch in chunked(note_ids, 500):
        out.extend(invoke("notesInfo", notes=batch))
    return out

def resolve_field_name(fields_dict, preferred, aliases):
    names = list(fields_dict.keys())
    if preferred in fields_dict:
        return preferred
    folded = {n.casefold(): n for n in names}
    if preferred.casefold() in folded:
        return folded[preferred.casefold()]
    for alias in aliases:
        if alias in fields_dict:
            return alias
        if alias.casefold() in folded:
            return folded[alias.casefold()]
    return None

def normalize_lemma(raw):
    text = TAG_RE.sub("", raw or "")
    return re.sub(r"\s+", " ", text).strip()

def normalize_subtitle(raw):
    text = TAG_RE.sub("", raw or "")
    text = re.sub(r"\s+", " ", text).strip()
    return text[:120]

def sanitize_link_title(text):
    return (text or "").replace("[", "(").replace("]", ")").replace("|", "/").strip()

def make_link(title, nid):
    return f"[{sanitize_link_title(title)}|nid{nid}]"

print("Connected AnkiConnect version:", invoke("version"))

In [ ]:
# 1) Load all new cards for grouping + only unsuspended new cards as candidates
all_new_card_ids = invoke("findCards", query=NEW_QUERY)
all_new_cards_info = get_cards_info(all_new_card_ids)
all_new_note_ids = sorted({c["note"] for c in all_new_cards_info})
all_new_notes_info = get_notes_info(all_new_note_ids)
new_note_by_id = {n["noteId"]: n for n in all_new_notes_info}

eligible_new_card_ids = invoke("findCards", query=NEW_ELIGIBLE_QUERY)
eligible_new_cards_info = get_cards_info(eligible_new_card_ids)
eligible_note_ids = {c["note"] for c in eligible_new_cards_info}

lemma_to_note_ids = defaultdict(set)
note_to_lemma = {}
missing_lemma_field = 0
detected_lemma_fields = defaultdict(int)

for note in all_new_notes_info:
    nid = note["noteId"]
    fields = note.get("fields", {})
    lemma_field = resolve_field_name(fields, LEMMA_FIELD_PREFERRED, LEMMA_FIELD_ALIASES)
    if not lemma_field:
        missing_lemma_field += 1
        continue

    detected_lemma_fields[lemma_field] += 1
    lemma = normalize_lemma(fields[lemma_field].get("value", ""))
    if lemma:
        note_to_lemma[nid] = lemma
        lemma_to_note_ids[lemma].add(nid)

duplicate_lemmas = {lemma: sorted(nids) for lemma, nids in lemma_to_note_ids.items() if len(nids) > 1}

print({
    "all_new_cards_found": len(all_new_card_ids),
    "all_new_notes_found": len(all_new_note_ids),
    "eligible_new_cards_found": len(eligible_new_card_ids),
    "eligible_new_notes_found": len(eligible_note_ids),
    "new_notes_missing_lemma_field": missing_lemma_field,
    "detected_lemma_fields": dict(detected_lemma_fields),
    "duplicate_lemma_groups": len(duplicate_lemmas),
})

In [ ]:
# 2) Build suspend/tag/link updates for duplicate groups
cards_to_suspend = []
notes_to_tag = set()
link_updates = []
groups_linked = 0
preview = []

for lemma, member_ids in duplicate_lemmas.items():
    eligible_member_ids = [nid for nid in member_ids if nid in eligible_note_ids]
    if not eligible_member_ids:
        continue

    groups_linked += 1

    # Titles for every member in this duplicate group
    member_titles = {}
    for idx, target_nid in enumerate(member_ids, start=1):
        note = new_note_by_id.get(target_nid)
        fields = note.get("fields", {}) if note else {}
        subtitle_field = resolve_field_name(fields, SUBTITLE_FIELD_PREFERRED, SUBTITLE_FIELD_ALIASES)
        subtitle = normalize_subtitle(fields[subtitle_field].get("value", "")) if subtitle_field else ""

        title = f"{lemma}_{idx}"
        if subtitle:
            title = f"{title}: {subtitle}"
        member_titles[target_nid] = title

    # For each source note, link to all siblings
    for source_nid in eligible_member_ids:
        source_note = new_note_by_id.get(source_nid)
        source_fields = source_note.get("fields", {}) if source_note else {}
        link_field_name = resolve_field_name(source_fields, LINK_FIELD, [LINK_FIELD.lower()]) or LINK_FIELD

        links = []
        for target_nid in member_ids:
            if target_nid == source_nid:
                continue
            links.append(make_link(member_titles[target_nid], target_nid))

        link_updates.append({"id": source_nid, "fields": {link_field_name: "<br>".join(links)}})
        notes_to_tag.add(source_nid)

        if len(preview) < 20:
            preview.append({
                "noteId": source_nid,
                "lemma": lemma,
                "links_count": len(links),
            })

# Suspend only the new cards whose notes are duplicates
duplicate_note_ids = set(notes_to_tag)
for c in eligible_new_cards_info:
    if c["note"] in duplicate_note_ids:
        cards_to_suspend.append(c["cardId"])

cards_to_suspend = sorted(set(cards_to_suspend))
notes_to_tag = sorted(notes_to_tag)

# Ensure Link field exists for note types involved
affected_models = sorted({
    (new_note_by_id[nid].get("modelName") if new_note_by_id.get(nid) else None)
    for nid in notes_to_tag
    if new_note_by_id.get(nid)
})
affected_models = [m for m in affected_models if m]
models_missing_link = []

for model in affected_models:
    field_names = invoke("modelFieldNames", modelName=model)
    if LINK_FIELD not in field_names:
        models_missing_link.append(model)
        if not DRY_RUN:
            invoke("modelFieldAdd", modelName=model, fieldName=LINK_FIELD)

print({
    "duplicate_lemma_groups": len(duplicate_lemmas),
    "cards_to_suspend": len(cards_to_suspend),
    "notes_to_tag": len(notes_to_tag),
    "tag": TAG_DUPLICATE_NEW,
    "affected_note_types": len(affected_models),
    "note_types_missing_link_field": models_missing_link,
    "sibling_groups_linked": groups_linked,
    "notes_with_link_updates": len(link_updates),
    "dry_run": DRY_RUN,
})

print("Preview (first 20 duplicates):")
for row in preview:
    print(row)

if not DRY_RUN:
    for batch in chunked(cards_to_suspend, 500):
        invoke("suspend", cards=batch)
    for batch in chunked(notes_to_tag, 500):
        invoke("addTags", notes=batch, tags=TAG_DUPLICATE_NEW)
    for upd in link_updates:
        invoke("updateNoteFields", note=upd)
    print("Applied suspend + tag + link updates.")
else:
    print("DRY_RUN=True, no changes written.")

## Apply

1. Run all cells with `DRY_RUN = True`.
2. Check `duplicate_lemma_groups`, `cards_to_suspend`, and preview.
3. Set `DRY_RUN = False` and rerun the last two code cells.

Make an Anki backup before applying.